# Examples Classification Ledger

This result-saved notebook classifies the current `examples/` tree into migration lanes before pruning. It is synchronized with `examples_classification_results.json`.

`protected_*` and "保護参照あり" are blockers, not destinations. The final homes are `docs`, `src`, `validation_test`, `panels`, or distill-delete after the lesson is recorded. Long-lived docs should point to docs artifacts, and executable code should point to `src` APIs or `validation_test` surfaces rather than `examples`.

In [1]:
from pathlib import Path
import json

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'examples_classification_results.json').exists():
    NOTEBOOK_DIR = Path('docs/examples_classification').resolve()
data = json.loads((NOTEBOOK_DIR / 'examples_classification_results.json').read_text(encoding='utf-8'))
print(json.dumps(data['summary'], ensure_ascii=False, indent=2))


{
  "total_examples_files": 1210,
  "tracked_examples_files": 1210,
  "total_python_files": 543,
  "topics": 11,
  "empty_topics": [],
  "route_counts": {
    "asset_or_fixture_review": 3,
    "distill_delete_review": 2,
    "docs_notebook_candidate": 167,
    "non_python_docs_or_asset_review": 99,
    "protected_mesh_or_cad_asset_review": 53,
    "protected_validation_reference": 28,
    "research_docs_or_distill_review": 86,
    "result_artifact_review": 419,
    "src_api_candidate": 22,
    "src_api_or_comment_reference_review": 7,
    "symbolic_derivation_review": 78,
    "unclassified_existing_review": 27,
    "validation_benchmark_candidate": 40,
    "validation_test_candidate": 179
  },
  "topic_recommendation_counts": {
    "new_topic_review": 1,
    "split_validation_test_first_then_docs_prune": 6,
    "protected_until_validation_references_are_migrated": 4
  },
  "protected_validation_reference_files": 28,
  "validation_test_candidates": 179,
  "validation_benchmark_candidate

## Migration Progress

This section records completed cleanup batches after the initial classification. Counts here come from `git ls-files -z -- examples` and reflect the current worktree, including tracked assets with non-ASCII names.

In [2]:
from IPython.display import Markdown, display

snapshot = data['current_worktree_examples_snapshot']
lines = [
    '| metric | value |',
    '| --- | ---: |',
    f"| tracked existing examples files | {snapshot['tracked_existing_files']} |",
    f"| tracked Python files | {snapshot['tracked_python_files']} |",
    f"| tracked JSON files | {snapshot['tracked_json_files']} |",
    f"| tracked notebooks | {snapshot['tracked_ipynb_files']} |",
    f"| tracked topic directories | {snapshot['tracked_topics']} |",
    f"| migration batches recorded | {len(data.get('migration_log', []))} |",
]
display(Markdown('\n'.join(lines)))

lines = ['| batch | removed from examples | moved topics |', '| --- | ---: | --- |']
for row in data.get('migration_log', []):
    topics = ', '.join(row.get('moved_topics', {}).keys())
    lines.append(f"| {row['batch_id']} | {row['examples_files_removed_from_examples']} | {topics} |")
display(Markdown('\n'.join(lines)))


| metric | value |
| --- | ---: |
| tracked existing examples files | 1210 |
| tracked Python files | 543 |
| tracked JSON files | 286 |
| tracked notebooks | 7 |
| tracked topic directories | 11 |
| migration batches recorded | 4 |

| batch | removed from examples | moved topics |
| --- | ---: | --- |
| 2026-06-28-small-docs-promotion-01 | 11 | peec_bema_convergence, figures, jou_translation_bench |
| 2026-06-28-small-docs-promotion-02 | 3 | root_gmsh_display, mathematica |
| 2026-06-28-solver-benchmarks-validation-03 | 28 | solver_benchmarks |
| 2026-06-28-induction-scattered-validation-04 | 2 | induction_heating/scattered_rhs_clean_test |

## Topic Recommendations

The table is ordered by Python-file count. `target_after_unblock` is the intended final home after examples references are rewritten.

In [3]:
from IPython.display import Markdown, display
rows = sorted(data['topics'], key=lambda r: (-r['python_files'], r['topic']))
lines = ['| topic | files | py | saved ipynb | refs block? | target after unblock | recommendation |', '| --- | --- | --- | --- | --- | --- | --- |']
for r in rows:
    targets = ', '.join(r.get('target_after_unblock', []))
    blocked = 'yes' if r.get('blocked_by_examples_reference') else 'no'
    lines.append(f"| {r['topic']} | {r['files']} | {r['python_files']} | {r['result_saved_docs_notebook_count']} | {blocked} | {targets} | {r['recommendation']} |")
display(Markdown('\n'.join(lines)))


| topic | files | py | saved ipynb | refs block? | target after unblock | recommendation |
| --- | --- | --- | --- | --- | --- | --- |
| maglev | 426 | 166 | 1 | yes | validation_test, docs, src, distill-delete | protected_until_validation_references_are_migrated |
| peec_integration | 218 | 130 | 5 | yes | validation_test, src, docs, distill-delete | split_validation_test_first_then_docs_prune |
| vim | 150 | 74 | 4 | yes | validation_test, src, docs | protected_until_validation_references_are_migrated |
| clebsch_hodograph | 119 | 46 | 2 | yes | validation_test, docs | split_validation_test_first_then_docs_prune |
| cubit_panels | 72 | 35 | 1 | yes | validation_test, docs, panels, distill-delete | split_validation_test_first_then_docs_prune |
| stream_function | 65 | 26 | 5 | yes | src, validation_test, docs | protected_until_validation_references_are_migrated |
| mixed_galerkin | 21 | 20 | 1 | no | src, validation_test, docs, distill-delete | split_validation_test_first_then_docs_prune |
| induction_heating | 27 | 19 | 3 | yes | src, validation_test, docs, panels | protected_until_validation_references_are_migrated |
| ngsolve_integration | 21 | 17 | 1 | no | docs, validation_test, src | split_validation_test_first_then_docs_prune |
| cube_uniform_field | 90 | 10 | 1 | yes | validation_test, docs | split_validation_test_first_then_docs_prune |
| README.md | 1 | 0 | 0 | no | review | new_topic_review |

## Agentic Reviews

Four explorer agents independently classified disjoint subtrees. The table records the ownership decision and the first migration moves, not deletion permission.

In [4]:
lines = ['| agent | scope | target after unblock | first moves |', '| --- | --- | --- | --- |']
for r in data['agentic_reviews']:
    scope = ', '.join(r['scope'])
    targets = ', '.join(r['target_after_unblock'])
    first = '<br>'.join(r['first_actions'][:3])
    lines.append(f"| {r['agent']} | {scope} | {targets} | {first} |")
display(Markdown('\n'.join(lines)))


| agent | scope | target after unblock | first moves |
| --- | --- | --- | --- |
| Sagan | maglev | validation_test, docs, src, distill-delete | Move research_cln/ngsolve_validation/dd_*.py and dd_full_pipeline.py to validation_test.<br>Review axifem_core.py for src API or deletion if superseded.<br>Move axifem/test_*.py to validation_test/axifem. |
| Bernoulli | peec_integration | validation_test, src, docs, distill-delete | Move benchmark_panel_vs_ngbem.py, vf_benchmark_same_data.py, and coupled validation scripts to validation_test.<br>Move validate_bemsibc.py, validate_bessel_sibc.py, validate_circular_coil_sibc.py, and validate_circular_wire_sibc.py.<br>Promote gmsh_centerline_reader.py and Cubit mesh generators/readers to src API or docs-local helpers. |
| Kierkegaard | vim, clebsch_hodograph, stream_function | validation_test, src, docs | Move VIM scripts directly imported by validation_test/feec, starting with bidirectional_map_* and foliated/hdiv_demag scripts.<br>Promote hdiv_vim_core_solve.py, multipole_moment_iter_scaling.py, and reference_yano_msc/generate_hex_mesh.py.<br>Treat validation_test/feec/test_clebsch_hodograph_research.py as the Clebsch migration driver. |
| Banach | cubit_panels, induction_heating, ngsolve_integration, mixed_galerkin, solver_benchmarks, cube_uniform_field, figures, jou_translation_bench, mathematica, peec_bema_convergence, mesh_generation | validation_test, src, docs, panels, distill-delete | Move induction_heating bem_inductance.py and clean_filament_test.py to validation_test while promoting BEM helpers to src.<br>Move solver_benchmarks/bench_peec_hacapk.py or update the smoke test first.<br>Move cube_uniform_field benchmark drivers and result corpora under validation/docs ownership. |

## Examples Reference Debt

These topics still have docs, validation, tests, packages, or source references pointing into `examples/`. Those references should migrate to docs artifacts, `src` APIs, `validation_test`, or panel assets before deletion.

In [5]:
rows = [r for r in data['topics'] if r.get('blocked_by_examples_reference')]
rows = sorted(rows, key=lambda r: (-sum((r.get('reference_root_counts') or {}).values()), r['topic']))
lines = ['| topic | reference roots | references | target after unblock | agentic note |', '| --- | --- | --- | --- | --- |']
for r in rows:
    roots = ', '.join(f"{k}:{v}" for k, v in sorted((r.get('reference_root_counts') or {}).items()))
    refs = sum((r.get('reference_root_counts') or {}).values())
    targets = ', '.join(r.get('target_after_unblock', []))
    note = r.get('agentic_note', '').replace('|', '/')
    lines.append(f"| {r['topic']} | {roots} | {refs} | {targets} | {note} |")
display(Markdown('\n'.join(lines)))


| topic | reference roots | references | target after unblock | agentic note |
| --- | --- | --- | --- | --- |
| peec_integration | docs:967 | 967 | validation_test, src, docs, distill-delete | Cleanup notebook is a routing handoff, not a completed migration. Move validation corpus first, then PEEC readers/generators to src and public demos to docs. |
| stream_function | docs:588, src:2, validation_test:1 | 591 | src, validation_test, docs | Extract reusable pieces from regcoil fusion and advanced planar FEM psi demos before moving direct validation refs; docs notebooks already exist and should become the public layer. |
| clebsch_hodograph | docs:473 | 473 | validation_test, docs | validation_test/feec/test_clebsch_hodograph_research.py imports the full corpus; treat it as the migration driver and rebuild docs after the validation move. |
| induction_heating | docs:214, src:3, validation_test:1 | 218 | src, validation_test, docs, panels | BEM helper code goes to src API, bem_inductance and clean filament checks go to validation_test, ESIM demos become docs notebooks, demoted panel journals stay panel assets. |
| vim | docs:14, src:8, validation_test:24 | 46 | validation_test, src, docs | validation_test/feec directly imports many scripts; move those executable surfaces first, then promote reusable HDiv/VIM helpers to src and keep docs notebooks as showcase. |
| maglev | docs:18, src:9, tests:7 | 34 | validation_test, docs, src, distill-delete | Split research_cln aggressively: DD/axisym/cuboid validation to validation_test, axifem_core to src or delete if superseded, Tanimoto/paper artifacts to docs, scratch iterations distill-delete. |
| cube_uniform_field | docs:8 | 8 | validation_test, docs | Benchmark drivers move to validation_test; JSON/PNG result corpus moves under docs or validation ownership. |
| cubit_panels | docs:2, src:2 | 4 | validation_test, docs, panels, distill-delete | Verification scripts belong in validation_test, scalar BIE SIBC belongs in docs notebook/source, Cubit CAD and journal assets move under their owning docs/panels/validation surface. |

## File Route Distribution

These are file-level lanes. A `protected_*` route means migration is blocked by an existing reference or protected asset owner; it is not a final home.

In [6]:
lines = ['| route | files | interpretation |', '| --- | --- | --- |']
for route, count in sorted(data['summary']['route_counts'].items(), key=lambda kv: (-kv[1], kv[0])):
    interpretation = 'temporary blocker' if route.startswith('protected_') else 'candidate lane/review'
    lines.append(f'| {route} | {count} | {interpretation} |')
display(Markdown('\n'.join(lines)))


| route | files | interpretation |
| --- | --- | --- |
| result_artifact_review | 419 | candidate lane/review |
| validation_test_candidate | 179 | candidate lane/review |
| docs_notebook_candidate | 167 | candidate lane/review |
| non_python_docs_or_asset_review | 99 | candidate lane/review |
| research_docs_or_distill_review | 86 | candidate lane/review |
| symbolic_derivation_review | 78 | candidate lane/review |
| protected_mesh_or_cad_asset_review | 53 | temporary blocker |
| validation_benchmark_candidate | 40 | candidate lane/review |
| protected_validation_reference | 28 | temporary blocker |
| unclassified_existing_review | 27 | candidate lane/review |
| src_api_candidate | 22 | candidate lane/review |
| src_api_or_comment_reference_review | 7 | candidate lane/review |
| asset_or_fixture_review | 3 | candidate lane/review |
| distill_delete_review | 2 | candidate lane/review |

## Next-Batch Pressure Points

This view groups actionable Python/code lanes. It helps choose whether the next batch should be validation migration, src API extraction, docs promotion, or distill-delete.

In [7]:
lines = ['| topic | validation/blocker | bench | src api | docs | delete review |', '| --- | --- | --- | --- | --- | --- |']
keys = ('protected_validation_reference', 'validation_test_candidate', 'validation_benchmark_candidate', 'src_api_candidate', 'src_api_or_comment_reference_review', 'docs_notebook_candidate', 'distill_delete_review')
for r in sorted(data['topics'], key=lambda row: -(sum(row['route_counts'].get(k, 0) for k in keys))):
    rc = r['route_counts']
    lines.append(f"| {r['topic']} | {rc.get('protected_validation_reference', 0) + rc.get('validation_test_candidate', 0)} | {rc.get('validation_benchmark_candidate', 0)} | {rc.get('src_api_candidate', 0) + rc.get('src_api_or_comment_reference_review', 0)} | {rc.get('docs_notebook_candidate', 0)} | {rc.get('distill_delete_review', 0)} |")
display(Markdown('\n'.join(lines)))


| topic | validation/blocker | bench | src api | docs | delete review |
| --- | --- | --- | --- | --- | --- |
| maglev | 142 | 4 | 4 | 1 | 1 |
| peec_integration | 31 | 10 | 14 | 64 | 1 |
| clebsch_hodograph | 0 | 7 | 0 | 39 | 0 |
| vim | 22 | 7 | 3 | 6 | 0 |
| stream_function | 1 | 0 | 2 | 23 | 0 |
| induction_heating | 1 | 1 | 4 | 13 | 0 |
| mixed_galerkin | 2 | 3 | 2 | 12 | 0 |
| ngsolve_integration | 2 | 1 | 0 | 8 | 0 |
| cube_uniform_field | 0 | 7 | 0 | 0 | 0 |
| cubit_panels | 6 | 0 | 0 | 1 | 0 |
| README.md | 0 | 0 | 0 | 0 | 0 |

In [8]:
assert data['schema'] == 'radia.docs.examples_classification.v1'
assert data['summary']['total_examples_files'] == len(data['files'])
assert data['summary']['topics'] == len(data['topics'])
assert data['summary']['total_python_files'] == sum(1 for row in data['files'] if row['suffix'] == '.py')
assert data['policy']['examples_references_are_temporary_blockers'] is True
assert len(data['agentic_reviews']) == data['summary']['agentic_review_count'] == 4
assert all('target_after_unblock' in row for row in data['topics'])
print('classification ledger OK: agentic reviews integrated; examples references are blockers, not destinations')


classification ledger OK: agentic reviews integrated; examples references are blockers, not destinations
